# 🏦 DeFi Risk Lab — Mô phỏng Rủi ro Thanh lý trong DeFi Lending
**Học phần:** Blockchain và Tài chính phi tập trung (DeFi)  
**Tên dự án:** DeFi Market Risk Simulator (DeFi Risk Lab)  
**Repository:** [github.com/tuananh21065-hub/defi-risk](https://github.com/tuananh21065-hub/defi-risk)

Notebook này cho phép bạn:
1. Thiết lập môi trường và chạy 17 unit test kiểm chứng mô hình
2. Chạy toàn bộ 5 nhóm kịch bản nghiên cứu (Sốc giá, Độ nhạy Gas, Cascade & Market Depth, Vốn Liquidator, Multi-seed)
3. Trực quan hóa dữ liệu qua biểu đồ trực quan (Matplotlib / Seaborn)
4. Sử dụng công cụ mô phỏng tương tác (Interactive Simulator với Widgets) trực tiếp trên Google Colab


## 1. ⚙️ Khởi tạo môi trường & Cài đặt thư viện


In [ ]:
import os
import sys

# Tự động clone repository nếu đang chạy trực tiếp trên Google Colab
if not os.path.exists("model.py"):
    print("Cloning repository từ GitHub...")
    !git clone https://github.com/tuananh21065-hub/defi-risk.git
    %cd defi-risk
else:
    print("Đang ở trong thư mục dự án.")

# Cài đặt các gói phụ thuộc cần thiết
!pip install -q pytest pandas matplotlib seaborn ipywidgets

print("✅ Môi trường đã sẵn sàng!")


## 2. 🧪 Kiểm thử đơn vị (Unit Tests)


In [ ]:
# Chạy 17 unit tests để xác minh tính đúng đắn của logic Health Factor, phân loại DEFAULTED vs REPAID, gas accounting
!pytest tests/ -v


## 3. 🚀 Chạy các kịch bản thực nghiệm chuẩn (main.py)


In [ ]:
# Chạy mô phỏng toàn bộ các nhóm kịch bản và xuất dữ liệu ra results/results.csv
!python -X utf8 main.py


## 4. 📊 Phân tích bảng dữ liệu kết quả thực nghiệm


In [ ]:
import pandas as pd

df = pd.read_csv("results/results.csv")

# Các cột quan trọng cần quan sát
key_cols = [
    "group", "scenario", "price_shock_pct", "gas_cost_usd", "liquidator_capital",
    "price_feedback", "after_n_touched", "after_n_repaid", "after_n_defaulted",
    "after_n_still_unsafe", "after_total_realized_bad_debt", "after_total_projected_deficit",
    "liquidator_profit", "final_eth_price"
]
actual_cols = [c for c in key_cols if c in df.columns]
display_df = df[actual_cols].copy()

# Định dạng hiển thị đẹp
display_df["price_shock_pct"] = display_df["price_shock_pct"].apply(lambda x: f"{x*100:.0f}%")
display_df["after_total_realized_bad_debt"] = display_df["after_total_realized_bad_debt"].apply(lambda x: f"${x:,.2f}")
display_df["liquidator_profit"] = display_df["liquidator_profit"].apply(lambda x: f"${x:,.2f}")

print("=== TỔNG HỢP CÁC KỊCH BẢN ĐÃ CHẠY ===")
display(display_df)


## 5. 📈 Trực quan hóa kết quả nghiên cứu (Biểu đồ)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from simulation import run_scenario
from scenarios import (
    generate_borrowers, INITIAL_ETH_PRICE, LIQUIDATOR_CAPITAL,
    PRICE_SHOCK_SCENARIOS, GAS_SENSITIVITY_SCENARIOS, CASCADE_SCENARIOS
)

sns.set_theme(style="whitegrid")
borrowers = generate_borrowers(n=50, eth_price=INITIAL_ETH_PRICE)

# --- BIỂU ĐỒ 1: Tác động của sốc giá ETH lên Nợ xấu & Vị thế thanh lý ---
fig, ax1 = plt.subplots(figsize=(9, 5))
shock_results = []
for s in PRICE_SHOCK_SCENARIOS:
    res = run_scenario(
        borrowers=borrowers,
        initial_eth_price=INITIAL_ETH_PRICE,
        price_shock_pct=s["price_shock_pct"],
        liquidator_capital=LIQUIDATOR_CAPITAL,
        gas_cost_usd=s["gas_cost_usd"]
    )
    shock_results.append({
        "shock": f"{s['price_shock_pct']*100:.0f}%",
        "touched": res["after"]["n_touched"],
        "defaulted": res["after"]["n_defaulted"],
        "bad_debt": res["after"]["total_realized_bad_debt"]
    })
df_shock = pd.DataFrame(shock_results)

color = "tab:blue"
ax1.set_xlabel("Mức sốc giá ETH (%)", fontsize=11, fontweight="bold")
ax1.set_ylabel("Số lượng Borrower", color=color, fontsize=11, fontweight="bold")
ax1.bar(df_shock["shock"], df_shock["touched"], alpha=0.6, color=color, label="Bị thanh lý (Touched)")
ax1.bar(df_shock["shock"], df_shock["defaulted"], alpha=0.9, color="darkred", label="Vỡ nợ (Defaulted)")
ax1.tick_params(axis="y", labelcolor=color)
ax1.legend(loc="upper left")

ax2 = ax1.twinx()
color = "tab:red"
ax2.set_ylabel("Realized Bad Debt ($)", color=color, fontsize=11, fontweight="bold")
ax2.plot(df_shock["shock"], df_shock["bad_debt"], color=color, marker="o", linewidth=2.5, label="Bad Debt ($)")
ax2.tick_params(axis="y", labelcolor=color)
ax2.legend(loc="upper center")

plt.title("Hình 1: Tác động của Sốc giá ETH đến Vị thế thanh lý và Nợ xấu thực tế", fontsize=13, fontweight="bold", pad=12)
plt.show()

# --- BIỂU ĐỒ 2: Liquidation Cascade - Diễn biến giá ETH theo Market Depth ---
plt.figure(figsize=(10, 5))
for s in CASCADE_SCENARIOS:
    depth = s.get("market_depth_usd") or 10_000_000.0
    res = run_scenario(
        borrowers=borrowers,
        initial_eth_price=INITIAL_ETH_PRICE,
        price_shock_pct=s["price_shock_pct"],
        liquidator_capital=LIQUIDATOR_CAPITAL,
        gas_cost_usd=s["gas_cost_usd"],
        price_feedback=s.get("price_feedback", False),
        market_depth_usd=depth
    )
    label = s["name"]
    if s.get("price_feedback"):
        label += f" (Depth: ${depth/1e6:.1f}M)"
    else:
        label += " (Không có feedback)"
    plt.plot(res["price_history"], marker="o", linewidth=2, label=label)

plt.title("Hình 2: Liquidation Spiral — Quỹ đạo giá ETH qua các vòng thanh lý", fontsize=13, fontweight="bold", pad=12)
plt.xlabel("Vòng thanh lý (Round)", fontsize=11, fontweight="bold")
plt.ylabel("Giá ETH ($)", fontsize=11, fontweight="bold")
plt.legend()
plt.tight_layout()
plt.show()

# --- BIỂU ĐỒ 3: Độ nhạy với chi phí Gas ---
gas_results = []
for s in GAS_SENSITIVITY_SCENARIOS:
    res = run_scenario(
        borrowers=borrowers,
        initial_eth_price=INITIAL_ETH_PRICE,
        price_shock_pct=s["price_shock_pct"],
        liquidator_capital=LIQUIDATOR_CAPITAL,
        gas_cost_usd=s["gas_cost_usd"]
    )
    gas_results.append({
        "gas_cost": f"${s['gas_cost_usd']:.0f}",
        "success": res["liquidator_stats"]["liquidation_count"],
        "failed_profit": res["liquidator_stats"]["failed_no_profit"],
        "still_unsafe": res["after"]["n_still_unsafe"]
    })
df_gas = pd.DataFrame(gas_results)

df_gas.plot(x="gas_cost", y=["success", "failed_profit", "still_unsafe"], kind="bar", figsize=(9, 5))
plt.title("Hình 3: Chi phí Gas ảnh hưởng đến hiệu quả thực thi thanh lý (Sốc -20%)", fontsize=13, fontweight="bold", pad=12)
plt.xlabel("Chi phí Gas mỗi giao dịch ($)", fontsize=11, fontweight="bold")
plt.ylabel("Số lượng giao dịch / vị thế", fontsize=11, fontweight="bold")
plt.legend(["Thanh lý thành công", "Bị hủy do không có lãi sau gas", "Vị thế vẫn nguy hiểm (Chưa xử lý)"])
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 6. 🎛️ Bộ mô phỏng tương tác trực tiếp (Interactive Simulator)


In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
from simulation import run_scenario
from scenarios import generate_borrowers

# Tạo các widget tương tác
w_n_borrowers = widgets.IntSlider(value=50, min=10, max=150, step=10, description="Borrowers:")
w_eth_price = widgets.FloatSlider(value=2000.0, min=1000.0, max=5000.0, step=100.0, description="Giá ETH ($):")
w_price_shock = widgets.IntSlider(value=-20, min=-50, max=0, step=5, description="Sốc giá (%):")
w_gas_cost = widgets.FloatSlider(value=5.0, min=1.0, max=100.0, step=1.0, description="Phí Gas ($):")
w_capital = widgets.FloatSlider(value=500000.0, min=50000.0, max=1000000.0, step=50000.0, description="Vốn Liq ($):")
w_feedback = widgets.Checkbox(value=False, description="Bật Price Feedback (Cascade)")
w_depth = widgets.Dropdown(
    options=[("Rất mỏng ($500K)", 500_000.0), ("Mỏng ($2M)", 2_000_000.0), ("Trung bình ($10M)", 10_000_000.0), ("Sâu ($50M)", 50_000_000.0)],
    value=10_000_000.0,
    description="Độ sâu thị trường:"
)
btn_run = widgets.Button(description="▶ Chạy mô phỏng", button_style="primary", icon="play")
out = widgets.Output()

def on_run_clicked(b):
    with out:
        clear_output()
        borrowers_pool = generate_borrowers(n=w_n_borrowers.value, eth_price=w_eth_price.value)
        res = run_scenario(
            borrowers=borrowers_pool,
            initial_eth_price=w_eth_price.value,
            price_shock_pct=w_price_shock.value / 100.0,
            liquidator_capital=w_capital.value,
            gas_cost_usd=w_gas_cost.value,
            price_feedback=w_feedback.value,
            market_depth_usd=w_depth.value
        )
        before = res["before"]
        after = res["after"]
        ls = res["liquidator_stats"]
        
        print("="*60)
        print(f"📊 KẾT QUẢ MÔ PHỎNG (Giá ETH cuối: ${after['eth_price']:,.2f})")
        print("="*60)
        print(f"• Số Borrower bị thanh lý: {after['n_touched']} / {after['n_borrowers']} ({after['touched_rate_pct']}%)")
        print(f"• Số Borrower vỡ nợ (Defaulted): {after['n_defaulted']}")
        print(f"• Số Borrower vẫn nguy hiểm (HF < 1): {after['n_still_unsafe']}")
        print(f"• Nợ xấu thực tế (Realized Bad Debt): ${after['total_realized_bad_debt']:,.2f}")
        print(f"• Nợ thâm hụt dự kiến (Projected Deficit): ${after['total_projected_deficit']:,.2f}")
        print(f"• Lợi nhuận của Liquidator: ${ls['total_profit']:,.2f}")
        print(f"• Số lần thanh lý bị bỏ qua vì Gas cao: {ls['failed_no_profit']}")
        print(f"• Số vòng mô phỏng đã chạy: {res['rounds_run']}")
        
        if w_feedback.value and len(res["price_history"]) > 1:
            plt.figure(figsize=(7, 3))
            plt.plot(res["price_history"], marker="o", color="crimson")
            plt.title("Đường suy giảm giá ETH qua các vòng thanh lý")
            plt.xlabel("Round")
            plt.ylabel("ETH Price ($)")
            plt.grid(True)
            plt.show()

btn_run.on_click(on_run_clicked)

ui = widgets.VBox([
    widgets.HBox([w_n_borrowers, w_eth_price]),
    widgets.HBox([w_price_shock, w_gas_cost]),
    widgets.HBox([w_capital, w_depth]),
    w_feedback,
    btn_run,
    out
])
display(ui)


## 7. 🌐 Tùy chọn: Chạy Streamlit Dashboard trên Google Colab
Nếu bạn muốn mở trực tiếp giao diện Web Streamlit từ Google Colab, hãy chạy cell bên dưới:


In [ ]:
# Chạy Streamlit nền và mở tunnel qua localtunnel
!pip install -q streamlit localtunnel
import urllib
print("Mật mã Endpoint IP (nhập vào trang localtunnel khi mở link):", urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip())

!streamlit run app.py --server.port 8501 & npx localtunnel --port 8501
